# Coleta dos dados da API

In [160]:
import requests
import json

import pandas as pd

## Funções para Query

In [161]:
URL = 'https://graphql.anilist.co'

In [162]:
def embedPageQuery(query:str):
    embededQuery =  '''
                    query ($page: Int, $perPage: Int) {
                        page: Page (page: $page, perPage: $perPage){
                    '''
    embededQuery += query
    embededQuery += "}}"
    
    return embededQuery

In [163]:
def queryPages(url:str, query:str, pageAmount:int=1, perPage:int=50):
    pages = []
    for currentPage in range(1, pageAmount+1):

        variables = {
            'page': currentPage,
            'perPage': perPage
        }

        print("Requisitando dados...")
        response = requests.post(url, json={'query': embedPageQuery(query), 'variables': variables})
        pages.append(json.loads(response.content)["data"]['page'])

    jointPages = []
    key = list(pages[0].keys())[0]
    for page in pages:
        jointPages += page[key]

    return jointPages

In [164]:
def getPages(url:str, query:str, name:str, pageAmount:int=1, perPage:int=50):
    try:
        with open(f'queryCache/{name}.json', 'r') as file:
            data = json.load(file)
            
        if data['query'] != query or data['entryAmount'] != pageAmount*perPage:
            raise FileNotFoundError
        
        pages = data['pages']
        print("Dados em cache!")

    except FileNotFoundError:
        pages = queryPages(url, query, pageAmount, perPage)

        data = {
            'pages': pages,
            'query': query,
            'entryAmount': pageAmount*perPage
        }

        with open(f'queryCache/{name}.json', 'w') as file:
            file.write(json.dumps(data, indent=4))

    return pages

## Requisições

In [165]:
mediaQuery = '''media (sort: SCORE_DESC, type: ANIME) {
                    id
                    title {
                        romaji
                        english
                        native
                    }
                    coverImage {
                        large
                    }
                    type
                    format
                    genres
                    status
                    description
                    startDate {
                        year
                        month
                        day
                    }
                    endDate {
                        year
                        month
                        day
                    }
                }
            '''

In [166]:
studioQuery =   '''studios (sort: FAVOURITES_DESC) {
                        id
                        name
                    }
                '''

In [167]:
charQuery = '''characters  (sort: FAVOURITES_DESC) {
                    id
                    name {
                        full
                    }
                    image {
                        large
                    }
                    description
                    gender
                    age
                }
            '''

In [168]:
staffQuery = '''staff (sort: FAVOURITES_DESC) {
                    id
                    image {
                        large
                    }
                    name {
                        full
                    }
                    description
                    gender
                    age
                    primaryOccupations
                }
            '''

In [169]:
userQuery = '''users (sort: WATCHED_TIME_DESC){
                    id
                    name
                    avatar {
                        large
                    }
                    createdAt
                }
            '''

In [170]:
queries = {
    'media':    (mediaQuery,    5),
    'studio':   (studioQuery,   1),
    'character':(charQuery,     5),
    'staff':    (staffQuery,    3),
    'user':     (userQuery,     5)
}

In [171]:
dfs = {}
for name, (query, pageAmount) in queries.items():
    print(f"\n-- {name.capitalize()} --")
    pages = getPages(URL, query, name, pageAmount)
    dfs[name] = pd.DataFrame(pages)


-- Media --
Dados em cache!

-- Studio --
Requisitando dados...

-- Character --
Dados em cache!

-- Staff --
Requisitando dados...
Requisitando dados...
Requisitando dados...

-- User --
Dados em cache!


In [172]:
dfs['media']

,id,title,coverImage,type,format,genres,status,description,startDate,endDate
0,154587,"{'romaji': 'Sousou no Frieren', 'english': 'Fr...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Adventure, Drama, Fantasy]",FINISHED,The adventure is over but life goes on for an ...,"{'year': 2023, 'month': 9, 'day': 29}","{'year': 2024, 'month': 3, 'day': 22}"
1,114129,"{'romaji': 'Gintama: THE FINAL', 'english': 'G...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,Two years have passed following the Tendoshuu'...,"{'year': 2021, 'month': 1, 'day': 8}","{'year': 2021, 'month': 1, 'day': 8}"
2,20996,"{'romaji': 'Gintama°', 'english': 'Gintama Sea...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Drama, Sci-Fi]",FINISHED,"Gintoki, Shinpachi, and Kagura return as the f...","{'year': 2015, 'month': 4, 'day': 8}","{'year': 2016, 'month': 3, 'day': 30}"
3,189046,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"In the deadly battle at Priestella, Subaru and...","{'year': 2026, 'month': 4, 'day': 8}","{'year': 2026, 'month': 9, 'day': 30}"
4,171627,"{'romaji': 'Chainsaw Man: Reze-hen', 'english'...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Action, Drama, Horror, Romance, Supernatural]",FINISHED,"Denji became “Chainsaw Man”, a boy with a devi...","{'year': 2025, 'month': 9, 'day': 19}","{'year': 2025, 'month': 9, 'day': 19}"
...,...,...,...,...,...,...,...,...,...,...
245,108632,{'romaji': 'Re:Zero kara Hajimeru Isekai Seika...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Adventure, Drama, Fantasy, Psychologi...",FINISHED,"Even after dying countless times, Subaru final...","{'year': 2020, 'month': 7, 'day': 8}","{'year': 2020, 'month': 9, 'day': 30}"
246,109403,"{'romaji': 'Non Non Biyori: Nonstop', 'english...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Comedy, Slice of Life]",FINISHED,Fresh excitement brews as someone new moves in...,"{'year': 2021, 'month': 1, 'day': 11}","{'year': 2021, 'month': 3, 'day': 29}"
247,140960,"{'romaji': 'SPY×FAMILY', 'english': 'SPY x FAM...",{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,TV,"[Action, Comedy, Slice of Life, Supernatural]",FINISHED,Everyone has a part of themselves they cannot ...,"{'year': 2022, 'month': 4, 'day': 9}","{'year': 2022, 'month': 6, 'day': 25}"
248,161474,{'romaji': 'Seishun Buta Yarou wa Randoseru Gi...,{'large': 'https://s4.anilist.co/file/anilistc...,ANIME,MOVIE,"[Drama, Psychological, Romance, Supernatural]",FINISHED,"March begins, and there is only one month left...","{'year': 2023, 'month': 12, 'day': 1}","{'year': 2023, 'month': 12, 'day': 1}"
